## AI-Powered Sustainable Multimodal Route Planner

### 1. Project Overview

This project develops an AI-powered sustainable multimodal route planner
that recommends feasible journeys based on a user's maximum budget,
maximum travel time, and available transportation modes.

The system generates routes using connected transportation segments and
allows different modes to be used across different segments of the same
journey. It evaluates total travel time, cost, and CO₂ emissions to
recommend suitable route options.

The project focuses on sustainable urban mobility and supports SDG 11:
Sustainable Cities and Communities, with SDG 13: Climate Action as a
secondary alignment.

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [16]:
df=pd.read_csv('multimodal_mobility_network.csv')
df

,connection_id,source,destination,mode,distance_km,duration_min,cost_inr,co2_kg
0,C001,Koti,Abids,Walking,2.0,26,0,0.0
1,C001,Koti,Abids,Bicycle,2.0,9,0,0.0
2,C001,Koti,Abids,Public Transport,2.0,16,15,0.2
3,C001,Koti,Abids,Car,2.0,8,35,0.5
4,C002,Koti,Nampally,Walking,3.0,38,0,0.0
...,...,...,...,...,...,...,...,...
115,C029,Tolichowki,Gachibowli,Car,5.0,17,85,1.0
116,C030,HITEC City,Kukatpally,Walking,7.0,89,0,0.0
117,C030,HITEC City,Kukatpally,Bicycle,7.0,30,0,0.0
118,C030,HITEC City,Kukatpally,Public Transport,7.0,50,25,0.7


In [17]:
df.head()

,connection_id,source,destination,mode,distance_km,duration_min,cost_inr,co2_kg
0,C001,Koti,Abids,Walking,2.0,26,0,0.0
1,C001,Koti,Abids,Bicycle,2.0,9,0,0.0
2,C001,Koti,Abids,Public Transport,2.0,16,15,0.2
3,C001,Koti,Abids,Car,2.0,8,35,0.5
4,C002,Koti,Nampally,Walking,3.0,38,0,0.0


#### data understanding and cleaning

In [18]:
print("Dataset Shape:", df.shape)

Dataset Shape: (120, 8)


In [19]:
print("Columns:")
print(df.columns.tolist())

Columns:
['connection_id', 'source', 'destination', 'mode', 'distance_km', 'duration_min', 'cost_inr', 'co2_kg']


In [20]:
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
connection_id    0
source           0
destination      0
mode             0
distance_km      0
duration_min     0
cost_inr         0
co2_kg           0
dtype: int64


In [21]:
print('Unique Connections: ',df['connection_id'].nunique())

Unique Connections:  30


In [30]:
locations=pd.unique(df[['source','destination']].values.ravel())
print('Unique locations: ',len(locations))

print('Locations: ')
print(locations)

Unique locations:  19
Locations: 
['Koti' 'Abids' 'Nampally' 'Charminar' 'Mehdipatnam' 'Ameerpet'
 'Banjara Hills' 'Begumpet' 'Secunderabad' 'Jubilee Hills' 'HITEC City'
 'Gachibowli' 'Miyapur' 'Kukatpally' 'Paradise' 'Dilsukhnagar' 'LB Nagar'
 'Uppal' 'Tolichowki']


In [31]:
print('Transportation Modes: ')
print(df['mode'].value_counts())

Transportation Modes: 
mode
Walking             30
Bicycle             30
Public Transport    30
Car                 30
Name: count, dtype: int64


#### Building the Connected Transportation Network

In [32]:
#unique connections
connections=df[['connection_id','source','destination']].drop_duplicates()
connections

,connection_id,source,destination
0,C001,Koti,Abids
4,C002,Koti,Nampally
8,C003,Koti,Charminar
12,C004,Abids,Nampally
16,C005,Abids,Charminar
20,C006,Nampally,Mehdipatnam
24,C007,Nampally,Ameerpet
28,C008,Mehdipatnam,Charminar
32,C009,Mehdipatnam,Banjara Hills
36,C010,Ameerpet,Begumpet


In [33]:
print('Total Connections: ',len(connections))

print('Total Locations: ',len(location))

Total Connections:  30
Total Locations:  10


In [65]:
### creating network structure

network={}

for index,row in connections.iterrows():
    source=row['source']
    destination=row['destination']
    
    if source not in network:
        network[source]=set()
        
    if destination not in network:
        network[destination]=set()
        
    network[source].add(destination)
    network[destination].add(source)

for location in network:
    network[location]=list(network[location])

In [68]:
### view the network

for location,connected_locations in network.items():
    print(location,'->',connected_locations)

Koti -> ['Charminar', 'Nampally', 'Uppal', 'Dilsukhnagar', 'Abids']
Abids -> ['Charminar', 'Nampally', 'Koti']
Nampally -> ['Mehdipatnam', 'Ameerpet', 'Abids', 'Koti']
Charminar -> ['Mehdipatnam', 'Abids', 'Koti']
Mehdipatnam -> ['Charminar', 'Nampally', 'Banjara Hills']
Ameerpet -> ['Kukatpally', 'Nampally', 'Begumpet', 'Banjara Hills']
Banjara Hills -> ['Jubilee Hills', 'Mehdipatnam', 'Ameerpet']
Begumpet -> ['Secunderabad', 'Jubilee Hills', 'Ameerpet', 'Paradise']
Secunderabad -> ['Paradise', 'Begumpet']
Jubilee Hills -> ['Gachibowli', 'HITEC City', 'Begumpet', 'Banjara Hills']
HITEC City -> ['Jubilee Hills', 'Gachibowli', 'Kukatpally', 'Miyapur']
Gachibowli -> ['Jubilee Hills', 'HITEC City', 'Tolichowki']
Miyapur -> ['HITEC City', 'Kukatpally']
Kukatpally -> ['HITEC City', 'Ameerpet', 'Miyapur']
Paradise -> ['Secunderabad', 'Begumpet']
Dilsukhnagar -> ['LB Nagar', 'Koti']
LB Nagar -> ['Uppal', 'Dilsukhnagar']
Uppal -> ['LB Nagar', 'Koti']
Tolichowki -> ['Gachibowli']


In [69]:
### verify all locations are connected

print('Locations in Network: ',len(network))
print('Locations in Dataset: ',len(locations))

Locations in Network:  19
Locations in Dataset:  19


In [70]:
missing_locations=set(locations)-set(network.keys())

print('Missing Locations: ',missing_locations)

Missing Locations:  set()


In [71]:
### checking direct connections

for location in network:
    print(f'{location}: {len(network[location])} direct connections')

Koti: 5 direct connections
Abids: 3 direct connections
Nampally: 4 direct connections
Charminar: 3 direct connections
Mehdipatnam: 3 direct connections
Ameerpet: 4 direct connections
Banjara Hills: 3 direct connections
Begumpet: 4 direct connections
Secunderabad: 2 direct connections
Jubilee Hills: 4 direct connections
HITEC City: 4 direct connections
Gachibowli: 3 direct connections
Miyapur: 2 direct connections
Kukatpally: 3 direct connections
Paradise: 2 direct connections
Dilsukhnagar: 2 direct connections
LB Nagar: 2 direct connections
Uppal: 2 direct connections
Tolichowki: 1 direct connections


In [72]:
###check for isolated locations

isolated_locations=[location for location in network if len(network[location])==0]

print('Isolated locations: ',isolated_locations)

Isolated locations:  []


In [77]:
print('Total Connections: ',len(connections))
print('Total Locations: ',len(network))
print('Isolated Locations: ',isolated_locations)

Total Connections:  30
Total Locations:  19
Isolated Locations:  []


####  Finding Possible Paths


In [78]:
### creating oath-finding function

def find_paths(network,source,destination):
    paths=[]
    
    def search(current,path):
        if current==destination:
            paths.append(path.copy())
            return 
        for next_location in network[current]:
            if  next_location not in path:
                path.append(next_location)
                search(next_location,path)
                path.pop()
    search(source,[source])
    
    return paths
        

### Test Basic path finding

In [81]:
paths=find_paths(network,'Koti','HITEC City')
print('Number of paths found: ',len(paths))

Number of paths found:  78


In [82]:
for i,path in enumerate(paths[:10],start=1):
    print(f'Path {i}:',"->".join(path))

Path 1: Koti->Charminar->Mehdipatnam->Nampally->Ameerpet->Kukatpally->HITEC City
Path 2: Koti->Charminar->Mehdipatnam->Nampally->Ameerpet->Kukatpally->Miyapur->HITEC City
Path 3: Koti->Charminar->Mehdipatnam->Nampally->Ameerpet->Begumpet->Jubilee Hills->Gachibowli->HITEC City
Path 4: Koti->Charminar->Mehdipatnam->Nampally->Ameerpet->Begumpet->Jubilee Hills->HITEC City
Path 5: Koti->Charminar->Mehdipatnam->Nampally->Ameerpet->Banjara Hills->Jubilee Hills->Gachibowli->HITEC City
Path 6: Koti->Charminar->Mehdipatnam->Nampally->Ameerpet->Banjara Hills->Jubilee Hills->HITEC City
Path 7: Koti->Charminar->Mehdipatnam->Banjara Hills->Jubilee Hills->Gachibowli->HITEC City
Path 8: Koti->Charminar->Mehdipatnam->Banjara Hills->Jubilee Hills->HITEC City
Path 9: Koti->Charminar->Mehdipatnam->Banjara Hills->Jubilee Hills->Begumpet->Ameerpet->Kukatpally->HITEC City
Path 10: Koti->Charminar->Mehdipatnam->Banjara Hills->Jubilee Hills->Begumpet->Ameerpet->Kukatpally->Miyapur->HITEC City


In [84]:
test_pairs=[('Koti','HITEC City'),('Koti','Charminar'),('Dilsukhnagar','Gachibowli')]

for source,destination in test_pairs:
    test_paths=find_paths(network,source,destination)
    
    print(f'{source}->{destination}')
    print('Number of paths found: ',len(test_paths))
    
    for i,path in enumerate(test_paths[:3],start=1):
        print(f' Path {i}:',"->".join(path))
        
    print()

Koti->HITEC City
Number of paths found:  78
 Path 1: Koti->Charminar->Mehdipatnam->Nampally->Ameerpet->Kukatpally->HITEC City
 Path 2: Koti->Charminar->Mehdipatnam->Nampally->Ameerpet->Kukatpally->Miyapur->HITEC City
 Path 3: Koti->Charminar->Mehdipatnam->Nampally->Ameerpet->Begumpet->Jubilee Hills->Gachibowli->HITEC City

Koti->Charminar
Number of paths found:  17
 Path 1: Koti->Charminar
 Path 2: Koti->Nampally->Mehdipatnam->Charminar
 Path 3: Koti->Nampally->Ameerpet->Kukatpally->HITEC City->Jubilee Hills->Banjara Hills->Mehdipatnam->Charminar

Dilsukhnagar->Gachibowli
Number of paths found:  200
 Path 1: Dilsukhnagar->LB Nagar->Uppal->Koti->Charminar->Mehdipatnam->Nampally->Ameerpet->Kukatpally->HITEC City->Jubilee Hills->Gachibowli
 Path 2: Dilsukhnagar->LB Nagar->Uppal->Koti->Charminar->Mehdipatnam->Nampally->Ameerpet->Kukatpally->HITEC City->Gachibowli
 Path 3: Dilsukhnagar->LB Nagar->Uppal->Koti->Charminar->Mehdipatnam->Nampally->Ameerpet->Kukatpally->Miyapur->HITEC City->Jubil

In [85]:
### verifying direcct path

direct_path=['Koti','Charminar']
print('Direct path exists: ',direct_path in find_paths(network,'Koti','Charminar'))

Direct path exists:  True


In [99]:
### Verifying Intermediate Path

paths_koti_hitech=find_paths(network,'Koti','HITEC City')

intermediate_paths=[
    path for path in paths_koti_hitech
    if len(path)>5
]

print('Intermediate paths found: ',len(intermediate_paths))

Intermediate paths found:  77
